In [11]:
import os
from openpyxl import load_workbook
from copy import copy

def copy_sheets_from_template(template_path, target_folder, selected_files, sheet_names):
    """
    Copy specified sheets from template to target Excel files
    """
    # Get files to process
    if not selected_files:
        files_to_process = [f for f in os.listdir(target_folder) if f.endswith('.xlsx')]
    else:
        files_to_process = selected_files
    
    print(f"Found {len(files_to_process)} files to process")
    
    # Load template
    template_wb = load_workbook(template_path)
    print(f"Template loaded with sheets: {template_wb.sheetnames}")
    
    for filename in files_to_process:
        file_path = os.path.join(target_folder, filename)
        print(f"\nProcessing: {filename}")
        
        try:
            target_wb = load_workbook(file_path)
            
            for sheet_name in sheet_names:
                if sheet_name in template_wb.sheetnames:
                    # Remove existing sheet if present
                    if sheet_name in target_wb.sheetnames:
                        del target_wb[sheet_name]
                    
                    # Copy entire sheet using openpyxl's built-in method
                    source_sheet = template_wb[sheet_name]
                    target_sheet = target_wb.create_sheet(sheet_name)
                    
                    # Copy all content and formatting
                    copy_worksheet(source_sheet, target_sheet)
                    
                    print(f"  ✓ Copied: {sheet_name}")
                else:
                    print(f"  ✗ Sheet not found: {sheet_name}")
            
            target_wb.save(file_path)
            target_wb.close()
            print(f"✓ Completed: {filename}")
            
        except Exception as e:
            print(f"✗ Error with {filename}: {e}")
    
    template_wb.close()
    print("\nAll files processed!")

def copy_worksheet(source, target):
    """Copy entire worksheet with all properties"""
    # Copy all cells
    for row in source.iter_rows():
        for cell in row:
            new_cell = target.cell(
                row=cell.row, 
                column=cell.column, 
                value=cell.value
            )
            if cell.has_style:
                new_cell.font = copy(cell.font)
                new_cell.border = copy(cell.border)
                new_cell.fill = copy(cell.fill)
                new_cell.number_format = cell.number_format
                new_cell.alignment = copy(cell.alignment)
    
    # Copy merged cells
    for merged_range in source.merged_cells.ranges:
        target.merge_cells(str(merged_range))
    
    # Copy dimensions
    for idx, row_dim in source.row_dimensions.items():
        target.row_dimensions[idx].height = row_dim.height
    
    for idx, col_dim in source.column_dimensions.items():
        target.column_dimensions[idx].width = col_dim.width
    
    # Copy all other properties
    target.sheet_view = copy(source.sheet_view)
    target.page_setup = copy(source.page_setup)
    target.page_margins = copy(source.page_margins)
    target.print_options = copy(source.print_options)
    
    # Copy auto filter
    if source.auto_filter.ref:
        target.auto_filter.ref = source.auto_filter.ref
    
    # Copy print titles and area
    target.print_title_rows = source.print_title_rows
    target.print_title_cols = source.print_title_cols
    target.print_area = source.print_area
    
    # Copy sheet properties
    target.sheet_properties.tabColor = source.sheet_properties.tabColor
    target.protection = copy(source.protection)
    
    # Copy header and footer
    target.oddHeader = source.oddHeader
    target.oddFooter = source.oddFooter
    target.evenHeader = source.evenHeader
    target.evenFooter = source.evenFooter
    target.firstHeader = source.firstHeader
    target.firstFooter = source.firstFooter

# Configuration
TEMPLATE_PATH = r"C:\MyDocuments\03Business\05ResearchAndAnalysis\StockInvestments\QuarterResultsScreenerExcels\Template\20251029_TCS_Modified.xlsx"
TARGET_FOLDER = r"C:\MyDocuments\03Business\05ResearchAndAnalysis\StockInvestments\QuarterResultsScreenerExcels\2026Q2"

# Parameters
selected_files = ['TCS_FY26Q2.xlsx']  # Empty list for all files
sheet_names = ["Assumptions&Scenario", "HistoricalData", "Model", "ModelBFSI"]

# Run the function
if __name__ == "__main__":
    copy_sheets_from_template(TEMPLATE_PATH, TARGET_FOLDER, selected_files, sheet_names)

Found 1 files to process
Template loaded with sheets: ['QuarterP&L', 'AnnualResults', 'BalanceSheet', 'CashFlow', 'SegmentAnalysis', 'ValuationHistory', 'ValuaitonModel', 'Assumptions&Scenario', 'HistoricalData', 'Model', 'ModelBFSI', 'AnalystReco', 'DBInsert', 'History&Ratio', 'Forecast', 'Reports', 'Customization', 'Data Sheet']

Processing: TCS_FY26Q2.xlsx
✗ Error with TCS_FY26Q2.xlsx: property 'sheet_view' of 'Worksheet' object has no setter

All files processed!


In [ ]:
import os
import xlwings as xw

TEMPLATE_PATH = r"C:\MyDocuments\03Business\05ResearchAndAnalysis\StockInvestments\QuarterResultsScreenerExcels\Template\20251029_TCS_Modified.xlsx"
TARGET_FOLDER = r"C:\MyDocuments\03Business\05ResearchAndAnalysis\StockInvestments\QuarterResultsScreenerExcels\2026Q2"

# Empty list => apply to all .xlsx files in TARGET_FOLDER
selected_files = ['TCS_FY26Q2.xlsx', 'INFY_FY26Q2.xlsx']
# selected_files = []  # uncomment to apply to all .xlsx files

# Template sheets you want copied
sheet_names = [
    "Assumptions&Scenario",
    "HistoricalData",
    "Model",
    "ModelBFSI",
]

overwrite_existing_sheets = True


def get_target_files(folder, selected_files):
    if selected_files:
        return selected_files
    return [
        f for f in os.listdir(folder)
        if f.lower().endswith(".xlsx") and not f.startswith("~$")
    ]


def main():
    # ⚠️ Strongly recommended: ensure you have a backup of 2026Q2 folder
    files_to_process = get_target_files(TARGET_FOLDER, selected_files)

    # Start Excel (invisible)
    app = xw.App(visible=False)
    app.display_alerts = False  # avoid "Are you sure you want to delete this sheet?" prompts

    try:
        tpl_wb = app.books.open(TEMPLATE_PATH)
        tpl_sheet_names = [sh.name for sh in tpl_wb.sheets]

        # Ensure the requested sheets actually exist in the template
        sheets_to_copy = [s for s in sheet_names if s in tpl_sheet_names]

        print("Template file:", TEMPLATE_PATH)
        print("Template sheets:", tpl_sheet_names)
        print("Sheets to copy from template:", sheets_to_copy)
        print("Target folder:", TARGET_FOLDER)
        print("Files to process:", files_to_process)

        for fname in files_to_process:
            target_path = os.path.join(TARGET_FOLDER, fname)
            if not os.path.isfile(target_path):
                print(f"\nSkipping (not found): {target_path}")
                continue

            print(f"\nProcessing: {target_path}")
            wb = app.books.open(target_path)
            try:
                # Current sheet names in target workbook
                def current_names():
                    return [sh.name for sh in wb.sheets]

                for sname in sheets_to_copy:
                    names_now = current_names()

                    # Delete existing sheet if overwrite is enabled
                    if sname in names_now and overwrite_existing_sheets:
                        print(f"  Deleting existing sheet: {sname}")
                        wb.sheets[sname].delete()
                        names_now = current_names()

                    # If sheet still exists and overwrite is False, skip
                    if sname in names_now:
                        print(f"  Sheet exists and overwrite=False, skipping: {sname}")
                        continue

                    # Copy from template
                    print(f"  Copying sheet from template: {sname}")
                    src_sheet = tpl_wb.sheets[sname]
                    # Use Excel's native Copy — this preserves everything
                    src_sheet.api.Copy(After=wb.sheets[-1].api)

                    # The newly copied sheet becomes the last sheet
                    new_sheet = wb.sheets[-1]
                    new_sheet.name = sname

                wb.save()
                print("  Saved.")
            finally:
                wb.close()
    finally:
        tpl_wb.close()
        app.quit()


if __name__ == "__main__":
    main()


Template file: C:\MyDocuments\03Business\05ResearchAndAnalysis\StockInvestments\QuarterResultsScreenerExcels\Template\20251029_TCS_Modified.xlsx
Template sheets: ['QuarterP&L', 'AnnualResults', 'BalanceSheet', 'CashFlow', 'SegmentAnalysis', 'ValuationHistory', 'ValuaitonModel', 'Assumptions&Scenario', 'HistoricalData', 'Model', 'ModelBFSI', 'AnalystReco', 'DBInsert', 'History&Ratio', 'Forecast', 'Reports', 'Customization', 'Data Sheet']
Sheets to copy from template: ['Assumptions&Scenario', 'HistoricalData', 'Model', 'ModelBFSI']
Target folder: C:\MyDocuments\03Business\05ResearchAndAnalysis\StockInvestments\QuarterResultsScreenerExcels\2026Q2
Files to process: ['TCS_FY26Q2.xlsx', 'INFY_FY26Q2.xlsx']

Processing: C:\MyDocuments\03Business\05ResearchAndAnalysis\StockInvestments\QuarterResultsScreenerExcels\2026Q2\TCS_FY26Q2.xlsx
  Copying sheet from template: Assumptions&Scenario
  Copying sheet from template: HistoricalData
  Copying sheet from template: Model
  Copying sheet from temp